# Multi-Peak Deconvolution with MultiGaussianFitter

When two or more γ-ray lines overlap, their contributions cannot be separated by simple integration. `MultiGaussianFitter` fits a sum of Gaussian peaks simultaneously using a non-linear least-squares optimiser (`scipy.optimize.curve_fit`), returning the peak positions, widths, and amplitudes with their full covariance matrix.

The covariance matrix enables rigorous uncertainty propagation: by sampling the fitted multivariate normal distribution, a Monte Carlo curve ensemble is built that gives a pointwise uncertainty band on the fitted function.

## Workflow
1. Load and calibrate the ¹⁵²Eu spectrum
2. Extract the overlapping-peak domain and subtract background
3. Fit with `MultiGaussianFitter`
4. Visualise the fit and its Monte Carlo uncertainty band

In [ ]:
from scispectrum.core import Spectrum, Domain
from scispectrum.calibration import AxisCalibration, ResolutionCalibration
from scispectrum.calibration.detector_calibration import DetectorCalibration
from scispectrum.identification import Convolution
from scispectrum.identification.snr import SNRFinder
from scispectrum.identification.kernels.mexican_hat import gaussian_2_dev

from scispectrum.domain_fitting import MultiGaussianFitter
from scispectrum.domain_analysis.find_peaks import find_domain_peaks
from scispectrum.domain_analysis.background import domain_erf_background
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load and calibrate the spectrum

> **Adapt:** replace `path`, `domains`, and `energies` with values for your own source and detector.

In [ ]:
# ── Adapt to your source and detector ────────────────────────────────────────
path     = '../Library/152Eu_calsource_10cm_85ks.txt'
domains  = [(383, 403), (1140, 1165), (1485, 1505), (2627, 2650), (3260, 3285), (4770, 4810)]
energies = [121.78, 344.28, 443.96, 778.9, 964.08, 1408.0]   # keV
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
spectrum = pd.read_csv(path, names=['counts'])
spectrum['channel'] = spectrum.index

axis_calibration = AxisCalibration(np.poly1d([0.29239695, 7.10512435]), "energy")
spectrum = Spectrum.from_dataframe(spectrum, axis_calib=axis_calibration)

calib = DetectorCalibration(spectrum=spectrum, peak_domains=domains, known_axis_values=energies)
energy_calibration, resolution_calibration = calib.generate()
spectrum.set_axis_calibration(energy_calibration)
spectrum.set_resolution_calibration(resolution_calibration)

## 2. Extract domain and subtract background

The 1436–1470 keV region of ¹⁵²Eu contains overlapping lines from ¹⁵²Sm at 1408 keV and ¹⁵²Gd at 1457 keV. The erf step background models the local Compton continuum.

> **Adapt** `start_val` and `stop_val` to the energy window containing your overlapping peaks.

In [ ]:
overlap       = spectrum.domain(start_val=1436, stop_val=1470)
background    = domain_erf_background(overlap)
subtracted    = overlap.subtract_background(background)

overlap.data.plot(label='original')
subtracted.data.plot(label='background subtracted')
plt.title('¹⁵²Eu — overlapping peaks 1436–1470 keV')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.legend()
plt.grid(True, which='both')
plt.tight_layout()
plt.show()

## 3. Fit overlapping peaks

`MultiGaussianFitter` automatically detects the number of peaks via prominence-based detection on the smoothed domain, then fits the full sum of Gaussians simultaneously. The result Dataset contains:
- `params` — fitted amplitudes, FWHMs, and centres (shape `n_peaks × 3`)
- `covariance` — full parameter covariance matrix (shape `n_params × n_params`)

> **Adapt `prominence`** if not all peaks are detected, or if noise spikes are picked up as peaks.

In [ ]:
gf       = MultiGaussianFitter()
fit_parm = gf.fit(subtracted, prominence=20)
fit_values = gf.evaluate(subtracted.axis, fit_parm)

print(f"Detected {len(fit_parm['i'])} peaks")
print("Centres [keV]:", fit_parm['params'].sel(quantity='center').values.round(3))

In [ ]:
subtracted.data.plot(label='data')
plt.plot(subtracted.axis, fit_values, label='fit', lw=2)
plt.title('¹⁵²Eu — deconvolution of overlapping peaks')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.grid(True, which='both')
for c in fit_parm['params'].sel(quantity='center').values:
    plt.axvline(x=c, ls='--', alpha=0.5, label=f'{c:.2f} keV')
plt.legend(fontsize='small')
plt.tight_layout()
plt.show()

In [ ]:
subtracted_overlap.data.plot()
plt.plot(subtracted_overlap.axis, fit_values)
plt.grid(True, which='both')
plt.title('Eu152 spectrum - specific overlapping peaks')
plt.xlabel('energy')
plt.ylabel('counts')

for c in fit_parm['params'].sel(quantity='center').values:
    plt.axvline(x=c)

## 4. Monte Carlo uncertainty band

`sample_curves` draws `size` parameter vectors from the multivariate normal defined by the fit covariance and evaluates the model at each. The resulting ensemble gives a pointwise 1σ uncertainty band that accounts for the full correlation structure of the fit parameters.

> **Adapt `size`:** larger → smoother band, slower. 500–2000 is typically sufficient.

In [ ]:
# ── Adapt sample size ─────────────────────────────────────────────────────────
sample_size = 1000
# ─────────────────────────────────────────────────────────────────────────────

fit_dist = gf.sample_curves(subtracted.axis, fit_parm, size=sample_size)

In [ ]:
mean_fit = fit_dist.mean(axis=0)
std_fit = fit_dist.std(axis=0)

subtracted_overlap.data.plot(yscale='log', label='domain')
plt.plot(subtracted_overlap.axis, mean_fit, label='mean')
plt.fill_between(subtracted_overlap.axis,
                 y1=mean_fit + std_fit,
                 y2=mean_fit - std_fit,
                 alpha=0.5, color='gray', label='1 std. dev.')

plt.grid(True, which='both')
plt.title('Eu152 spectrum - specific overlapping peaks')
plt.xlabel('energy')
plt.ylabel('counts')
plt.ylim([1, mean_fit.max()])
plt.legend()
for c in fit_parm['params'].sel(quantity='center').values:
    plt.axvline(x=c, color='black', ls='--')